# 4. Geographic Analysis
Map schools by location and analyze regional distributions and performance.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../scripts')
from helpers import load_education_data

df = load_education_data('../data/khi_education_demo.csv')
df_latest = df[df['Year'] == df['Year'].max()].reset_index(drop=True)
sns.set_style('whitegrid')

## Basic Geographic Distribution

In [ ]:
# Schools by region
region_dist = df_latest.groupby('Region').agg({
    'School_Name': 'count',
    'Latitude': 'mean',
    'Longitude': 'mean',
    'Overall_Score': 'mean',
    'Enrollment': 'sum'
}).round(2)
region_dist.rename(columns={'School_Name': 'Schools'}, inplace=True)

print("Distribution by Region (Latest Year):")
print(region_dist)

In [ ]:
# Schools by district
district_dist = df_latest.groupby('District').agg({
    'School_Name': 'count',
    'Overall_Score': 'mean',
    'Enrollment': 'sum',
    'Attendance_Rate': 'mean'
}).round(2)
district_dist.rename(columns={'School_Name': 'Schools'}, inplace=True)

print("\nDistribution by District:")
print(district_dist.sort_values('Overall_Score', ascending=False))

## Scatter Map: Geographic Visualization

In [ ]:
# Create scatter plot to show school locations colored by performance
fig, ax = plt.subplots(figsize=(12, 8))

scatter = ax.scatter(
    df_latest['Longitude'], 
    df_latest['Latitude'],
    c=df_latest['Overall_Score'],
    s=df_latest['Enrollment']/10,  # Size proportional to enrollment
    cmap='RdYlGn',
    alpha=0.7,
    edgecolors='black',
    linewidth=1.5
)

# Add school labels
for idx, row in df_latest.iterrows():
    ax.annotate(row['School_Name'][:10], 
                (row['Longitude'], row['Latitude']),
                fontsize=7, alpha=0.7)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Schools Geographic Distribution (2024)\nColor = Performance, Size = Enrollment', fontsize=12, fontweight='bold')

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Overall Score', rotation=270, labelpad=20)

plt.tight_layout()
plt.show()

## Regional Performance Comparison

In [ ]:
# Regional performance metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Schools per region
region_counts = df_latest.groupby('Region')['School_Name'].count().sort_values(ascending=False)
axes[0, 0].bar(region_counts.index, region_counts.values, color='skyblue')
axes[0, 0].set_title('Number of Schools by Region')
axes[0, 0].set_ylabel('Count')

# Average score by region
region_scores = df_latest.groupby('Region')['Overall_Score'].mean().sort_values(ascending=False)
axes[0, 1].bar(region_scores.index, region_scores.values, color='lightcoral')
axes[0, 1].set_title('Average Score by Region')
axes[0, 1].set_ylabel('Score')

# Total enrollment by region
region_enroll = df_latest.groupby('Region')['Enrollment'].sum().sort_values(ascending=False)
axes[1, 0].bar(region_enroll.index, region_enroll.values, color='lightgreen')
axes[1, 0].set_title('Total Enrollment by Region')
axes[1, 0].set_ylabel('Students')

# Average attendance by region
region_attend = df_latest.groupby('Region')['Attendance_Rate'].mean().sort_values(ascending=False)
axes[1, 1].bar(region_attend.index, region_attend.values, color='lightyellow')
axes[1, 1].set_title('Average Attendance by Region')
axes[1, 1].set_ylabel('Attendance %')

plt.tight_layout()
plt.show()

## Heatmap: District Analysis

In [ ]:
# Create pivot table for heatmap
df_all_years = df.groupby(['District', 'Year']).agg({
    'Overall_Score': 'mean',
    'Enrollment': 'sum',
    'Attendance_Rate': 'mean'
}).round(2).reset_index()

pivot_score = df_all_years.pivot(index='District', columns='Year', values='Overall_Score')
pivot_enroll = df_all_years.pivot(index='District', columns='Year', values='Enrollment')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Performance heatmap
sns.heatmap(pivot_score, annot=True, fmt='.1f', cmap='RdYlGn', ax=axes[0], cbar_kws={'label': 'Overall Score'})
axes[0].set_title('Performance Heatmap: Average Score by District and Year')

# Enrollment heatmap
sns.heatmap(pivot_enroll, annot=True, fmt='.0f', cmap='Blues', ax=axes[1], cbar_kws={'label': 'Total Enrollment'})
axes[1].set_title('Enrollment Heatmap: Total Students by District and Year')

plt.tight_layout()
plt.show()

## Key Geographic Insights

In [ ]:
print("KEY GEOGRAPHIC INSIGHTS:")
print("=" * 50)
print(f"\nHighest Performing Region: {region_scores.idxmax()} ({region_scores.max():.2f})")
print(f"Lowest Performing Region: {region_scores.idxmin()} ({region_scores.min():.2f})")
print(f"\nMost Schools: {region_counts.idxmax()} ({region_counts.max()} schools)")
print(f"Highest Enrollment: {region_enroll.idxmax()} ({region_enroll.max():.0f} students)")
print(f"\nBest Attendance: {region_attend.idxmax()} ({region_attend.max():.2f}%)")
print(f"Lowest Attendance: {region_attend.idxmin()} ({region_attend.min():.2f}%)")